<a href="https://colab.research.google.com/github/9RED9/GPT-Agent/blob/main/00_deep_agent_custom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

제공해주신 `deepagents` 라이브러리 활용 코드를 잘 확인했습니다. 이 코드는 LangGraph/LangChain 기반의 고수준 에이전트 생성기인 `deepagents`를 사용하여, 특정 역할(LangChain 문서 어시스턴트)이 부여된 에이전트를 만들고 질문에 대한 답변을 도출하는 깔끔한 로직을 담고 있습니다.

구글 코랩(Google Colab) 환경에서 백지상태부터 곧바로 실행하실 수 있도록 **의존성 설치**와 **API 키 입력** 단계를 포함한 완전한 코드로 재구성했습니다.

한 가지 바로잡아 드리자면, LangChain의 메시지 객체는 결과값에 접근할 때 `.text` 대신 **`.content`** 속성을 사용해야 정상적으로 작동합니다. 이 부분을 코드에 반영해 수정해 두었습니다.

---

### 💻 구글 코랩(Google Colab) 실행용 코드

코랩 환경에서 새로운 셀을 열고 아래 코드를 그대로 복사하여 붙여넣기 한 뒤 실행하시면 됩니다.

In [3]:
# 1. 필수 라이브러리 설치 (최초 실행 시 약간의 시간이 소요될 수 있습니다)
!pip install -qU deepagents langchain-core langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.1/236.1 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 64.4 MB/s eta 0:00:00


In [4]:


# 2. 환경 변수 설정 (OpenAI API 키 입력)
import os
import getpass

# API 키가 환경변수에 없다면 안전하게 입력받도록 처리합니다.
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요: ")


OpenAI API Key를 입력하세요: ··········
[{'id': 'rs_0feccf479438cdc4006a5829a44d44819a8518a25b74719517', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqWCm0Gx0No_HoDBn81ZUGMwPSpY4Wh1gRSIQzXSz5a0Hwl_KDyU6z_5175r9dWInmavvfBVLYBNaDlcgAqx6ecrIyyixlaMZJQc0hSy358yBC5yggCzbPIhw9A1xmakRyYojKPxtTv64hJ-ZzZIyWkdSiykl1mZTJTg_dGM-wlkqEgBByqcpVwrz_xz5DmY0HxicBxOgifxfvwxI8n8IZC3HEPoW01WNoMnvZXTv1gs-9OHCGRDRqGs7CKqchZ0Bw0RA7YiFoUojINaDQuXEj6CJzyTCmeGUqO7PBKpT1IFp3aV1Egzb6bgZiV0L3ftS-PgszExwV6hAuWBKpU7_brbMn3bxtqIgRE8ltn49TqeE3_J9yRr8BLr45deBhVif3wdnVt35Q9CKSGWuo5OMjRSKbDw7m8xrwhHR79BZIY92NKqqrDEArKwQrUHCSlCtseq3PnR6h86DRDqO3-GHVdUrHqul9YfyfiW5ADT44-gk5I_oDr6zeQ1F92aWRwTo6fnAv3lqNdfra-P2kLaEhxZpelyGsuuosz5JDyXW4q9DYhy6UjW1poVlv7sPWpEZzkdVbDzSLKcjafeZAeH2kQCxQyYf1djSYNwr-cqBJulStMIqHYBWH_hsX4mgd2B9voSjHyT_7iSBU3fYadJC_fUag7t7Szxhm_vD8Z74-5DLNaLJkSP7Q5IT6n-NvnGGjVdrS3SomYmEFnK_FzUMzelQLov5lGV4DpHeN4id_E46DVTrTgxmYyJI027G7mg573kW7qfC-zxIry3X2Zb6Vy8d8eKcYMJ_UdG-fD8iw2JkWQRWfYmp7

In [ ]:


# 2. 환경 변수 설정 (OpenAI API 키 입력)
import os
import getpass

# API 키가 환경변수에 없다면 안전하게 입력받도록 처리합니다.
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요: ")

# 3. 에이전트 모듈 임포트 및 코드 실행
from deepagents import create_deep_agent
from langchain_core.messages import HumanMessage

EXAMPLE_QUERY = "How do I stream intermediate tool results from a subagent?"

# 기준 에이전트(baseline_agent) 초기화
baseline_agent = create_deep_agent(
    model="openai:gpt-5.5",
    tools=[],
    system_prompt=(
        "You are a helpful LangChain documentation assistant. "
        "Answer questions about LangChain APIs and patterns."
    ),
)

# 에이전트 실행하여 결과 받아오기
result = baseline_agent.invoke(
    {"messages": [HumanMessage(content=EXAMPLE_QUERY)]}
)

# 결과 텍스트 출력 (오류 방지를 위해 .text 대신 .content 사용)
print(result["messages"][-1].content)

OpenAI API Key를 입력하세요: ··········
[{'id': 'rs_0feccf479438cdc4006a5829a44d44819a8518a25b74719517', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqWCm0Gx0No_HoDBn81ZUGMwPSpY4Wh1gRSIQzXSz5a0Hwl_KDyU6z_5175r9dWInmavvfBVLYBNaDlcgAqx6ecrIyyixlaMZJQc0hSy358yBC5yggCzbPIhw9A1xmakRyYojKPxtTv64hJ-ZzZIyWkdSiykl1mZTJTg_dGM-wlkqEgBByqcpVwrz_xz5DmY0HxicBxOgifxfvwxI8n8IZC3HEPoW01WNoMnvZXTv1gs-9OHCGRDRqGs7CKqchZ0Bw0RA7YiFoUojINaDQuXEj6CJzyTCmeGUqO7PBKpT1IFp3aV1Egzb6bgZiV0L3ftS-PgszExwV6hAuWBKpU7_brbMn3bxtqIgRE8ltn49TqeE3_J9yRr8BLr45deBhVif3wdnVt35Q9CKSGWuo5OMjRSKbDw7m8xrwhHR79BZIY92NKqqrDEArKwQrUHCSlCtseq3PnR6h86DRDqO3-GHVdUrHqul9YfyfiW5ADT44-gk5I_oDr6zeQ1F92aWRwTo6fnAv3lqNdfra-P2kLaEhxZpelyGsuuosz5JDyXW4q9DYhy6UjW1poVlv7sPWpEZzkdVbDzSLKcjafeZAeH2kQCxQyYf1djSYNwr-cqBJulStMIqHYBWH_hsX4mgd2B9voSjHyT_7iSBU3fYadJC_fUag7t7Szxhm_vD8Z74-5DLNaLJkSP7Q5IT6n-NvnGGjVdrS3SomYmEFnK_FzUMzelQLov5lGV4DpHeN4id_E46DVTrTgxmYyJI027G7mg573kW7qfC-zxIry3X2Zb6Vy8d8eKcYMJ_UdG-fD8iw2JkWQRWfYmp7

**💡 실행 팁:**

* 코드 실행 시 입력창이 나타나면 준비하신 `sk-...`로 시작하는 **OpenAI API Key**를 붙여넣고 엔터를 누르시면 됩니다. (입력 시 보안을 위해 화면에 문자가 표시되지 않습니다.)
* 만약 `gpt-5.5` 모델에 대한 접근 권한이 없다면, `model="openai:gpt-4o"` 등으로 모델명을 변경하여 테스트해 보시는 것을 권장합니다.

In [ ]:
# 2. 환경 변수 설정 (OpenAI API 키 입력)
import os
from google.colab import userdata

# API 키가 환경변수에 없다면 Colab Secrets에서 가져오도록 처리합니다.
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

OpenAI API Key를 입력하세요: ··········
[{'id': 'rs_0feccf479438cdc4006a5829a44d44819a8518a25b74719517', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqWCm0Gx0No_HoDBn81ZUGMwPSpY4Wh1gRSIQzXSz5a0Hwl_KDyU6z_5175r9dWInmavvfBVLYBNaDlcgAqx6ecrIyyixlaMZJQc0hSy358yBC5yggCzbPIhw9A1xmakRyYojKPxtTv64hJ-ZzZIyWkdSiykl1mZTJTg_dGM-wlkqEgBByqcpVwrz_xz5DmY0HxicBxOgifxfvwxI8n8IZC3HEPoW01WNoMnvZXTv1gs-9OHCGRDRqGs7CKqchZ0Bw0RA7YiFoUojINaDQuXEj6CJzyTCmeGUqO7PBKpT1IFp3aV1Egzb6bgZiV0L3ftS-PgszExwV6hAuWBKpU7_brbMn3bxtqIgRE8ltn49TqeE3_J9yRr8BLr45deBhVif3wdnVt35Q9CKSGWuo5OMjRSKbDw7m8xrwhHR79BZIY92NKqqrDEArKwQrUHCSlCtseq3PnR6h86DRDqO3-GHVdUrHqul9YfyfiW5ADT44-gk5I_oDr6zeQ1F92aWRwTo6fnAv3lqNdfra-P2kLaEhxZpelyGsuuosz5JDyXW4q9DYhy6UjW1poVlv7sPWpEZzkdVbDzSLKcjafeZAeH2kQCxQyYf1djSYNwr-cqBJulStMIqHYBWH_hsX4mgd2B9voSjHyT_7iSBU3fYadJC_fUag7t7Szxhm_vD8Z74-5DLNaLJkSP7Q5IT6n-NvnGGjVdrS3SomYmEFnK_FzUMzelQLov5lGV4DpHeN4id_E46DVTrTgxmYyJI027G7mg573kW7qfC-zxIry3X2Zb6Vy8d8eKcYMJ_UdG-fD8iw2JkWQRWfYmp7

In [ ]:


# 2. 환경 변수 설정 (OpenAI API 키 입력)
import os
import getpass

# API 키가 환경변수에 없다면 안전하게 입력받도록 처리합니다.
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요: ")

# 3. 에이전트 모듈 임포트 및 코드 실행
from deepagents import create_deep_agent
from langchain_core.messages import HumanMessage

EXAMPLE_QUERY = "How do I stream intermediate tool results from a subagent?"

# 기준 에이전트(baseline_agent) 초기화
baseline_agent = create_deep_agent(
    model="openai:gpt-5.5",
    tools=[],
    system_prompt=(
        "You are a helpful LangChain documentation assistant. "
        "Answer questions about LangChain APIs and patterns."
    ),
)

# 에이전트 실행하여 결과 받아오기
result = baseline_agent.invoke(
    {"messages": [HumanMessage(content=EXAMPLE_QUERY)]}
)

# 결과 텍스트 출력 (오류 방지를 위해 .text 대신 .content 사용)
print(result["messages"][-1].content)

OpenAI API Key를 입력하세요: ··········
[{'id': 'rs_0feccf479438cdc4006a5829a44d44819a8518a25b74719517', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqWCm0Gx0No_HoDBn81ZUGMwPSpY4Wh1gRSIQzXSz5a0Hwl_KDyU6z_5175r9dWInmavvfBVLYBNaDlcgAqx6ecrIyyixlaMZJQc0hSy358yBC5yggCzbPIhw9A1xmakRyYojKPxtTv64hJ-ZzZIyWkdSiykl1mZTJTg_dGM-wlkqEgBByqcpVwrz_xz5DmY0HxicBxOgifxfvwxI8n8IZC3HEPoW01WNoMnvZXTv1gs-9OHCGRDRqGs7CKqchZ0Bw0RA7YiFoUojINaDQuXEj6CJzyTCmeGUqO7PBKpT1IFp3aV1Egzb6bgZiV0L3ftS-PgszExwV6hAuWBKpU7_brbMn3bxtqIgRE8ltn49TqeE3_J9yRr8BLr45deBhVif3wdnVt35Q9CKSGWuo5OMjRSKbDw7m8xrwhHR79BZIY92NKqqrDEArKwQrUHCSlCtseq3PnR6h86DRDqO3-GHVdUrHqul9YfyfiW5ADT44-gk5I_oDr6zeQ1F92aWRwTo6fnAv3lqNdfra-P2kLaEhxZpelyGsuuosz5JDyXW4q9DYhy6UjW1poVlv7sPWpEZzkdVbDzSLKcjafeZAeH2kQCxQyYf1djSYNwr-cqBJulStMIqHYBWH_hsX4mgd2B9voSjHyT_7iSBU3fYadJC_fUag7t7Szxhm_vD8Z74-5DLNaLJkSP7Q5IT6n-NvnGGjVdrS3SomYmEFnK_FzUMzelQLov5lGV4DpHeN4id_E46DVTrTgxmYyJI027G7mg573kW7qfC-zxIry3X2Zb6Vy8d8eKcYMJ_UdG-fD8iw2JkWQRWfYmp7